In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
multicare_df = pd.read_parquet("Datasets/MutiCaRe.parquet")

In [3]:
multicare_df.head()

,article_id,case_id,case_text
0,PMC3738355,PMC3738355_01,A 53-year-old woman presented with a 10-year h...
1,PMC5015624,PMC5015624_01,A 69-year-old Caucasian female with coronary a...
2,PMC6381877,PMC6381877_01,A 60-year-old male smoker presented with persi...
3,PMC5912312,PMC5912312_01,A 41-year-old female with a past medical histo...
4,PMC5912312,PMC5912312_02,A 51-year-old male with a history of SCAD pres...


In [6]:
multicare_df.iloc[10]["case_text"]

'A 34-year-old female who was 38 weeks pregnant presented with atypical chest pain for 1 day. Electrocardiogram showed ST-segment elevations in the anterolateral leads with a negative first troponin level. Coronary angiogram showed SCAD of the mid portion of the LAD with large intramural hematoma compromising the blood flow to the first and second diagonal branches of the LAD with TIMI grade 3 flow into distal vessel (Figure 7). No intervention was done and she was admitted to the cardiac care unit where a conservative approach was adopted. After undergoing Cesarean section, repeat coronary catheterization was performed. It showed no evidence of intramural hematoma and a healed LAD dissection.'

In [17]:
drug_bank_df = pd.read_csv("Datasets/drugbank_clean.csv", low_memory=True)

/var/folders/1x/36nz44_569501n4xs0px2_8m0000gn/T/ipykernel_32402/891165286.py:1: DtypeWarning: Columns (29,30,31,32,36,37) have mixed types. Specify dtype option on import or set low_memory=False.
  drug_bank_df = pd.read_csv("Datasets/drugbank_clean.csv", low_memory=True)


In [18]:
drug_bank_df.head()

,type,created,updated,drugbank-id,name,description,cas-number,unii,state,groups,...,reactions,snp-effects,snp-adverse-drug-reactions,targets,carriers,transporters,fda-label,msds,average-mass,monoisotopic-mass
0,biotech,2005-06-13,2023-01-03,DB00001,Lepirudin,Lepirudin is a recombinant hirudin formed by 6...,138068-37-8,Y43GF64R34,solid,approved withdrawn,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,BE0000048,NaN,NaN,NaN,NaN,NaN,NaN
3,biotech,2005-06-13,2023-01-03,DB00002,Cetuximab,Cetuximab is a recombinant chimeric human/mous...,205923-56-4,PQX0D8J21J,liquid,approved,...,NaN,NaN,NaN,NaN,NaN,NaN,//s3-us-west-2.amazonaws.com/drugbank/fda_labe...,//s3-us-west-2.amazonaws.com/drugbank/msds/DB0...,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,P08637 P12318,NaN,BE0002096 BE0000767 BE0000710 BE0002097 BE0002...,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
drug_bank_df = drug_bank_df.dropna(how='all')

In [22]:
# Focus on safety profile columns
safety_columns = ['drugbank-id', 'name', 'groups', 'toxicity', 
                  'drug-interactions', 'food-interactions', 
                  'snp-adverse-drug-reactions']

In [23]:
safety_df = drug_bank_df[safety_columns].copy()

# Check data availability
print("Data availability in safety columns:")
for col in ['toxicity', 'drug-interactions', 'food-interactions', 'snp-adverse-drug-reactions']:
    non_null = safety_df[col].notna().sum()
    total = len(safety_df)
    print(f"{col}: {non_null}/{total} ({non_null/total*100:.1f}%)")

# View drugs with complete safety data
complete_safety = safety_df.dropna(subset=['toxicity', 'drug-interactions'])
print(f"\nDrugs with toxicity AND drug-interaction data: {len(complete_safety)}")
complete_safety.head()

Data availability in safety columns:
toxicity: 2432/16761 (14.5%)
drug-interactions: 4478/16761 (26.7%)
food-interactions: 1378/16761 (8.2%)
snp-adverse-drug-reactions: 45/16761 (0.3%)

Drugs with toxicity AND drug-interaction data: 2069


,drugbank-id,name,groups,toxicity,drug-interactions,food-interactions,snp-adverse-drug-reactions
0,DB00001,Lepirudin,approved withdrawn,The acute toxicity of intravenous lepirudin wa...,DB13821 DB00328 DB08953 DB00682 DB00928 DB0912...,Avoid herbs and supplements with anticoagulant...,NaN
3,DB00002,Cetuximab,approved,The intravenous LD<sub>50</sub> is > 300 mg/kg...,DB05209 DB14905 DB06474 DB09381 DB06647 DB1195...,NaN,NaN
8,DB00006,Bivalirudin,approved investigational,"Based on a study by Gleason et al., the no-obs...",DB13821 DB00328 DB08953 DB00682 DB00928 DB0912...,Avoid herbs and supplements with anticoagulant...,NaN
10,DB00007,Leuprolide,approved investigational,"Leuprolide is considered extremely safe, with ...",DB13821 DB01345 DB00328 DB00682 DB00825 DB0120...,NaN,NaN
11,DB00008,Peginterferon alfa-2a,approved investigational,Peginterferon alfa-2a may manifest neuropsychi...,DB08880 DB00328 DB00682 DB00928 DB15274 DB1340...,Drink plenty of fluids.,NaN
